# Carga Precios, Bencina

In [99]:
import requests
import pandas as pd
import os
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots


url = "https://mindicador.cl/api/dolar/2026"
data = requests.get(url).json()
df = pd.DataFrame(data["serie"])
df.to_csv("../datasets/dolar_2026.csv", index=False)

In [100]:
origen = Path("../datasets/bencina_en_linea_comprimido")
destino = Path("../datasets/bencina_en_linea")
destino.mkdir(exist_ok=True)
for archivo in origen.glob("*.csv.bz2"):
    df = pd.read_csv(
        archivo,
        compression="bz2",
        encoding="latin1",
        engine="python",
        sep=None
    )
    salida = destino / archivo.name.replace(".csv.bz2", ".parquet")
    df.to_parquet(salida, index=False)

# Limpieza

In [101]:
df_2012 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2012.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad Herrera Bravo Ltda,COPEC,Avenida IrarrÃ¡zaval,5277.0,ÃuÃ±oa,Metropolitana,759,2012-01-01,Gasolina 93,"-33,4540062109273","-70,57525992393493"
1,pb630302,Distribuidora Dagnino Giacobbe y CÃ­a. Ltda. ...,Sin Bandera,Camino a Codegua Km,1.0,Chimbarongo,Gral. Bernardo O'Higgins,612,2012-12-13,Petroleo Diesel,"-34,71739081954427","-71,02941691875458"
2,co120501,SOCIEDAD COMERCIAL IBAÃ¯Â¿Â½EZ Y NEGRON LTDA.,COPEC,"PANAMERICANA NORTE KM 1750, EX S. V",0.0,Pozo Almonte,TarapacÃ¡,777,2012-12-06,Gasolina 95,"-21,09225805724834","-69,5928230881691"
3,te1312301,Gilberto Zamorano Vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,"-33,44420837761395","-70,59720039367676"
4,co1313201,PATRICIO REYES INFANTE Y CIA LTDA,COPEC,Av. Vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,"-33,38974288164123","-70,57051241397858"
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa Rosa,25.0,Puente Alto,Metropolitana,750,2012-06-29,Gasolina 93,"-33,61835394119413","-70,62618434429169"
6,sh510601,COMERCIALIZADORA DE COMBUSTIBLES ENERGIAS Y OT...,SHELL,Avda. Los Carrera,620.0,QuilpuÃ©,ValparaÃ­so,846,2012-08-16,Gasolina 97,"-33,04439263678347","-71,46056592464447"
7,co830501,Administradora de Ventas al Detalle Ltda. ...,COPEC,LASTARRIA,11.0,MulchÃ©n,BÃ­o BÃ­o,623,2012-06-11,Petroleo Diesel,"-37,71319890348465","-72,24748313426971"
8,co1311902,Villanueva y Clark Limitada,COPEC,Alberto Llona,640.0,MaipÃº,Metropolitana,782,2012-11-22,Gasolina 97,"-33,52131677600925","-70,75603008270264"
9,te1340106,ALONSO SPA,Sin Bandera,Panamericana Sur Km 17,12667.0,San Bernardo,Metropolitana,820,2012-03-15,Gasolina 93,"-33,56301455104863","-70,71177363395691"


In [102]:
df_2012.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251917 entries, 0 to 251916
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   251917 non-null  object 
 1   razon_social         251917 non-null  object 
 2   distribuidor         251917 non-null  object 
 3   direccion_calle      251917 non-null  object 
 4   direccion_numero     251917 non-null  float64
 5   comuna               251917 non-null  object 
 6   region               251917 non-null  object 
 7   precio               251917 non-null  int64  
 8   fecha_actualizacion  251917 non-null  object 
 9   combustible          251917 non-null  object 
 10  latitud              251917 non-null  object 
 11  longitud             251917 non-null  object 
dtypes: float64(1), int64(1), object(10)
memory usage: 23.1+ MB


## Latitud - Longitud

In [103]:
def limpiar_cordenada(coordenada):
    if ',' in coordenada:
        coordenada = coordenada.replace(",", ".")
    
    cardinalidad = ["N", "S", "E", "W", "n", "s", "e", "w"]
    for cardinal in cardinalidad:
        if coordenada.endswith(cardinal):
            coordenada = coordenada.replace(cardinal, "")
            break
    
    return coordenada

df_2012["latitud"] = df_2012["latitud"].apply(limpiar_cordenada).astype(float)
df_2012["longitud"] = df_2012["longitud"].apply(limpiar_cordenada).astype(float)

df_2012.tail()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
251912,co630103,COMERCIAL H Y M LTDA,COPEC,AV. B. OHIGGINS ESQ. M. VELASCO,0.0,San Fernando,Gral. Bernardo O'Higgins,825,2012-06-28,Gasolina 97,-34.584062,-70.983728
251913,co1320101,Arriagada y Mora Limitada,COPEC,Av. Concha y Toro,316.0,Puente Alto,Metropolitana,596,2012-12-20,Petroleo Diesel,-33.616764,-70.574214
251914,pe1410101,Sociedad Jorge Contreras CaÃ±as y Cia. Ltda.,PETROBRAS,Picarte,1027.0,Valdivia,Los RÃ­Â­os,645,2012-11-08,Petroleo Diesel,-39.817165,-73.234777
251915,te1312904,COMERCIAL E INVERSIONES DELGADO HERMANOS & CIA...,SHELL,Santa Rosa,2700.0,San JoaquÃ­n,Metropolitana,576,2012-12-19,Petroleo Diesel,-33.481286,-70.641280
251916,co1311302,Inversiones Cossio y Hutt Ltda.,COPEC,Avenida Ossa,591.0,La Reina,Metropolitana,853,2012-08-31,Gasolina 97,-33.448246,-70.571183


## Fecha

In [104]:
df_2012['fecha_actualizacion'] = pd.to_datetime(df_2012['fecha_actualizacion'], errors='coerce')
df_2012.head()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad Herrera Bravo Ltda,COPEC,Avenida IrarrÃ¡zaval,5277.0,ÃuÃ±oa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora Dagnino Giacobbe y CÃ­a. Ltda. ...,Sin Bandera,Camino a Codegua Km,1.0,Chimbarongo,Gral. Bernardo O'Higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,SOCIEDAD COMERCIAL IBAÃ¯Â¿Â½EZ Y NEGRON LTDA.,COPEC,"PANAMERICANA NORTE KM 1750, EX S. V",0.0,Pozo Almonte,TarapacÃ¡,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto Zamorano Vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,PATRICIO REYES INFANTE Y CIA LTDA,COPEC,Av. Vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512


## Texto

In [105]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return None
    valor = str(valor).strip()
    try:
        valor = valor.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        valor = valor
    return valor.capitalize()

df_2012['razon_social'] = df_2012['razon_social'].apply(limpiar_texto)
df_2012['comuna'] = df_2012['comuna'].apply(limpiar_texto)
df_2012['direccion_calle'] = df_2012['direccion_calle'].apply(limpiar_texto)
df_2012['region'] = df_2012['region'].apply(limpiar_texto)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277.0,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1.0,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0.0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808.0,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380.0,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa rosa,25.0,Puente alto,Metropolitana,750,2012-06-29,Gasolina 93,-33.618354,-70.626184
6,sh510601,Comercializadora de combustibles energias y ot...,SHELL,Avda. los carrera,620.0,Quilpué,Valparaíso,846,2012-08-16,Gasolina 97,-33.044393,-71.460566
7,co830501,Administradora de ventas al detalle ltda.,COPEC,Lastarria,11.0,Mulchén,Bío bío,623,2012-06-11,Petroleo Diesel,-37.713199,-72.247483
8,co1311902,Villanueva y clark limitada,COPEC,Alberto llona,640.0,Maipú,Metropolitana,782,2012-11-22,Gasolina 97,-33.521317,-70.756030
9,te1340106,Alonso spa,Sin Bandera,Panamericana sur km 17,12667.0,San bernardo,Metropolitana,820,2012-03-15,Gasolina 93,-33.563015,-70.711774


## direccion_numero

In [106]:
df_2012['direccion_numero'] = df_2012['direccion_numero'].astype(int)
df_2012.head(10)

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512
5,co1320104,Administradora de ventas al detalle ltda,COPEC,Santa rosa,25,Puente alto,Metropolitana,750,2012-06-29,Gasolina 93,-33.618354,-70.626184
6,sh510601,Comercializadora de combustibles energias y ot...,SHELL,Avda. los carrera,620,Quilpué,Valparaíso,846,2012-08-16,Gasolina 97,-33.044393,-71.460566
7,co830501,Administradora de ventas al detalle ltda.,COPEC,Lastarria,11,Mulchén,Bío bío,623,2012-06-11,Petroleo Diesel,-37.713199,-72.247483
8,co1311902,Villanueva y clark limitada,COPEC,Alberto llona,640,Maipú,Metropolitana,782,2012-11-22,Gasolina 97,-33.521317,-70.756030
9,te1340106,Alonso spa,Sin Bandera,Panamericana sur km 17,12667,San bernardo,Metropolitana,820,2012-03-15,Gasolina 93,-33.563015,-70.711774


In [107]:
df_2012['razon_social'].astype(str)
df_2012['comuna'].astype(str)
df_2012['direccion_calle'].astype(str)
df_2012['region'].astype(str)
df_2012['distribuidor'].astype(str)
df_2012['combustible'].astype(str)
df_2012.head()

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1312002,Sociedad herrera bravo ltda,COPEC,Avenida irarrázaval,5277,Ñuñoa,Metropolitana,759,2012-01-01,Gasolina 93,-33.454006,-70.575260
1,pb630302,Distribuidora dagnino giacobbe y cía. ltda. 7...,Sin Bandera,Camino a codegua km,1,Chimbarongo,Gral. bernardo o'higgins,612,2012-12-13,Petroleo Diesel,-34.717391,-71.029417
2,co120501,Sociedad comercial ibaï¿½ez y negron ltda.,COPEC,"Panamericana norte km 1750, ex s. v",0,Pozo almonte,Tarapacá,777,2012-12-06,Gasolina 95,-21.092258,-69.592823
3,te1312301,Gilberto zamorano vega,SHELL,Holanda,2808,Providencia,Metropolitana,790,2012-12-06,Gasolina 97,-33.444208,-70.597200
4,co1313201,Patricio reyes infante y cia ltda,COPEC,Av. vitacura,6380,Vitacura,Metropolitana,873,2012-09-06,Gasolina 97,-33.389743,-70.570512


In [108]:
df_2012.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 251917 entries, 0 to 251916
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   id                   251917 non-null  object        
 1   razon_social         251917 non-null  object        
 2   distribuidor         251917 non-null  object        
 3   direccion_calle      251917 non-null  object        
 4   direccion_numero     251917 non-null  int64         
 5   comuna               251917 non-null  object        
 6   region               251917 non-null  object        
 7   precio               251917 non-null  int64         
 8   fecha_actualizacion  251917 non-null  datetime64[ns]
 9   combustible          251917 non-null  object        
 10  latitud              251917 non-null  float64       
 11  longitud             251917 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(7)
memory usage: 23.1

In [109]:
df_2013 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2013.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2014 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2014.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2015 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2015.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2016 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2016.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)   
df_2017 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2017.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2018 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2018.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2019 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2019.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2020 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2020.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)      
df_2021 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2021.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2022 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2022.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)



In [110]:
df_2023 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2023.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)  
df_2024 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2024.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2025 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2025.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)
df_2026 = pd.read_csv("../datasets/bencina_en_linea_comprimido/2026.csv.bz2", compression="bz2", encoding="latin1", engine="python", sep=None)

In [111]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026]:
    print(df.columns.tolist())

['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud', 'longitud']
['id', 'razon_social', 'distribuidor', 'direccion_calle', 'direccion_numero', 'comuna', 'region', 'precio', 'fecha_actualizacion', 'combustible', 'latitud'

In [112]:
print(df_2023.columns.tolist())

['codigo', 'razon_social', 'distribuidor', 'direccion', 'latitud', 'longitud', 'nom_comuna', 'nom_region', 'combustible', 'precio', 'unidad_cobro', 'atencion', 'fecha_actualizacion', 'hora_actualizacion', 'es_electrolinera', 'es_gasolinera']


In [113]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022]:
    df['latitud'] = df['latitud'].apply(limpiar_cordenada).astype(float)
    df['longitud'] = df['longitud'].apply(limpiar_cordenada).astype(float)
    df['fecha_actualizacion'] = pd.to_datetime(df['fecha_actualizacion'], errors='coerce')
    df['razon_social'] = df['razon_social'].apply(limpiar_texto)
    df['comuna'] = df['comuna'].apply(limpiar_texto)
    df['direccion_calle'] = df['direccion_calle'].apply(limpiar_texto)
    df['region'] = df['region'].apply(limpiar_texto)
    df['direccion_numero'] = df['direccion_numero'].astype(int)  

In [114]:
cols_renombrar = {
    'codigo': 'id',
    'direccion': 'direccion_calle',
    'nom_comuna': 'comuna',
    'nom_region': 'region'
}

for df in [df_2023, df_2024, df_2025, df_2026]:
    df.rename(columns=cols_renombrar, inplace=True)
    df.drop(columns=['hora_actualizacion', 'unidad_cobro', 'atencion'], inplace=True)
    df.drop_duplicates(inplace=True)
    df['es_electrolinera'] = df['es_electrolinera'].astype(bool)
    df['es_gasolinera'] = df['es_gasolinera'].astype(bool)
    df['latitud'] = df['latitud'].apply(limpiar_cordenada).astype(float)
    df['longitud'] = df['longitud'].apply(limpiar_cordenada).astype(float)
    df['fecha_actualizacion'] = pd.to_datetime(df['fecha_actualizacion'], errors='coerce')
    df['razon_social'] = df['razon_social'].apply(limpiar_texto)
    df['comuna'] = df['comuna'].apply(limpiar_texto)
    df['direccion_calle'] = df['direccion_calle'].apply(limpiar_texto)
    df['region'] = df['region'].apply(limpiar_texto)
    df['precio'] = df['precio'].astype(int)

In [115]:
df_2025.head(10)

,id,razon_social,distribuidor,direccion_calle,latitud,longitud,comuna,region,combustible,precio,fecha_actualizacion,es_electrolinera,es_gasolinera
0,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1282,2024-12-20,False,True
1,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1310,2025-01-02,False,True
2,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1315,2025-01-08,False,True
3,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1344,2025-01-09,False,True
4,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1343,2025-01-23,False,True
5,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1372,2025-01-31,False,True
6,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1371,2025-02-14,False,True
7,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1349,2025-02-20,False,True
8,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1348,2025-02-27,False,True
9,co110104a,Ewald zippel y cï¿½a limitada,COPEC,Avda. salvador allende 2345,-20.256424,-70.128077,Iquique,Tarapacá,A93,1320,2025-03-13,False,True


## Duplicados

In [116]:
for df in [df_2013, df_2014, df_2015, df_2016, df_2017, df_2018, df_2019, df_2020, df_2021, df_2022, df_2023, df_2024, df_2025, df_2026]:
    df.sort_values('fecha_actualizacion', inplace=True)
    df.drop_duplicates(subset=['id', 'razon_social','combustible'], keep='last', inplace=True)

In [117]:
df_2025.head()

,id,razon_social,distribuidor,direccion_calle,latitud,longitud,comuna,region,combustible,precio,fecha_actualizacion,es_electrolinera,es_gasolinera
222909,co1312001,Inversiones pargar ltda,COPEC,Avenida irarrázaval 1102,-33.453098,-70.619141,Ñuñoa,Metropolitana de santiago,GNC,100,2012-01-12,False,True
98073,pb740103,Inversiones hn spa,HN,Avenida presidente ibañez 1175,-35.842647,-71.581807,Linares,Del maule,97,700,2012-03-22,False,True
29088,co410104,Kaptol ltda,COPEC,Balmaceda 1839,-29.917805,-71.253902,La serena,Coquimbo,GLP,0,2012-04-10,False,True
213729,co1312801,Granese y urrutia ltda.,COPEC,Américo vespucio esq. lo boza 2001,-33.386357,-70.758981,Renca,Metropolitana de santiago,KE,633,2012-04-12,False,True
100407,hn740701,Inversiones hn spa,HN,Abate molina 160,-35.676128,-71.743874,Villa alegre,Del maule,GNC,563,2012-05-02,False,True


In [118]:
os.makedirs('../datasets/bencina_limpia', exist_ok=True)

dataframes = {
    'df_2012': df_2012, 'df_2013': df_2013, 'df_2014': df_2014,
    'df_2015': df_2015, 'df_2016': df_2016, 'df_2017': df_2017,
    'df_2018': df_2018, 'df_2019': df_2019, 'df_2020': df_2020,
    'df_2021': df_2021, 'df_2022': df_2022, 'df_2023': df_2023,
    'df_2024': df_2024, 'df_2025': df_2025, 'df_2026': df_2026,
}

for nombre, df in dataframes.items():
    anio = nombre.split('_')[1]
    df.to_parquet(f'../datasets/bencina_limpia/{anio}.parquet', index=False)


# Archivos casen 2024

# Limpieza y extracción de datos: Ingreso CASEN 2024

Se procesa el archivo `Ingreso_Casen_2024.xlsx`

### Hojas que se usan

| Hoja | Contenido | CSV generado |
|------|-----------|-------------|
| 1 | Ingreso promedio nacional por tipo de ingreso | `ingreso_nacional.csv` |
| 3 | Ingreso promedio por región | `ingreso_region.csv` |
| 24 | Mínimo, media y máximo del ingreso per cápita por decil | `ingreso_percapita_decil.csv` |

In [119]:
import pandas as pd
from openpyxl import load_workbook
from pathlib import Path

excel_path = Path('../datasets/casen_2024/Ingreso_Casen_2024.xlsx')
path_carpeta = Path('../datasets/casen_limpia')
path_carpeta.mkdir(parents=True, exist_ok=True)

años = [2006, 2009, 2011, 2013, 2015, 2017, 2020, 2022, 2024]

wb = load_workbook(excel_path, read_only=True, data_only=True)
print(wb.sheetnames)
wb.close()

['INDICE', 'Notas Técnicas', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24']


## Ingreso promedio nacional

La hoja 1 tiene la siguiente estructura:
- Fila 9–12: estimaciones de ingreso (del trabajo, autónomo, subsidios, monetario)
- Columnas B–J (índices 1–9): ingreso en $ de cada año
- Columnas M–U (índices 12–20): ingreso en $ reales de noviembre 2024
- Filas 19–22: errores estándar de cada estimación

Cada fila es un tipo de ingreso y cada columna es un año.

In [120]:
wb = load_workbook(excel_path, read_only=True, data_only=True)
ws = wb['1']
filas = list(ws.iter_rows(values_only=True))
wb.close()

tipos_ingreso = ['Ingreso del trabajo','Ingreso autónomo','Subsidios monetarios','Ingreso monetario',]

# datos de estimación empiezan en la fila 9 (índice 0)
# errores estándar empiezan en la fila 19
estimacion = 9
errores = 19
registros = []
for i, tipo in enumerate(tipos_ingreso):
    fila_est = filas[estimacion + i]
    fila_ee  = filas[errores+ i]

    for j, anio in enumerate(años):
        registros.append({
            'anio':anio,
            'tipo_ingreso':tipo,
            'ingreso_nominal':fila_est[1 + j],   # col B en adelante
            'ingreso_real_nov2024':fila_est[12 + j],  # col M en adelante
            'error_estandar_nominal':fila_ee[1 + j],
            'error_estandar_real':fila_ee[12 + j],
        })

df_nacional = pd.DataFrame(registros)
df_nacional = df_nacional.dropna(subset=['ingreso_nominal'])

In [121]:
df_nacional.to_csv(path_carpeta / 'ingreso_nacional.csv', index=False, encoding='utf-8-sig')

Descripción de columnas — ingreso_nacional.csv

*   anio: Año en que se realizó la encuesta CASEN. Los años disponibles son 2006, 2009, 2011, 2013, 2015, 2017, 2020, 2022 y 2024.
*   tipo_ingreso: Tipo de ingreso medido. Puede ser uno de cuatro valores: Ingreso del trabajo, Ingreso autónomo, Subsidios monetarios o Ingreso monetario.
*   ingreso_nominal: Ingreso promedio mensual del hogar en pesos del año en que se midió. No es directamente comparable entre años distintos debido a la inflación.
*   ingreso_real_nov2024: Ingreso promedio mensual del hogar expresado en pesos de noviembre de 2024. Permite comparar el poder adquisitivo real entre distintos años.
*   error_estandar_nominal: Error estándar de la estimación del ingreso nominal. Refleja la precisión estadística de la encuesta al ser una muestra y no un censo.
*   error_estandar_real: Error estándar de la estimación del ingreso expresado en pesos de noviembre de 2024

Definiciones de cada tipo de ingreso (según notas del archivo original)

*   Ingreso del trabajo: Ingresos que obtienen todos los miembros del hogar, excluido el servicio doméstico puertas adentro, en su ocupación por concepto de sueldos y salarios, monetarios y en especies, ganancias provenientes del trabajo independiente y la auto-provisión de bienes producidos por el hogar.
*   Ingreso autónomo: Suma de todos los pagos que reciben todos los miembros del hogar, excluido el servicio doméstico puertas adentro, provenientes tanto del trabajo como de la propiedad de los activos. Incluye sueldos y salarios, ganancias del trabajo independiente, auto-provisión de bienes, rentas, intereses, dividendos, retiro de utilidades, jubilaciones, pensiones o montepíos y transferencias corrientes.
*   Subsidios monetarios: Aportes en dinero que reciben todos los miembros del hogar, excluido el servicio doméstico puertas adentro, del Estado a través de programas sociales.
*   Ingreso monetario: Suma del ingreso autónomo y los subsidios monetarios percibidos por todos los miembros del hogar, excluido el servicio doméstico puertas adentro.

Notas metodológicas

*   Los ingresos están corregidos por no respuesta.
*   La Casen en Pandemia 2020 se aplicó con cambios metodológicos asociados a su modalidad de aplicación, por lo que las comparaciones con años anteriores deben realizarse con resguardos.
*   Los factores de expansión se construyeron con proyecciones de población basadas en el Censo 2017, por lo que los valores pueden diferir de publicaciones anteriores.

Fuente: Ministerio de Desarrollo Social, Encuesta Casen.

## Ingreso promedio por región

La estructura es:
- Fila 9 en adelante: un bloque por cada tipo de ingreso (4 tipos × 17 filas = 68 filas de datos)
- Columna A: nombre del tipo de ingreso (solo en la primera fila del bloque)
- Columna B: nombre de la región
- Columnas C–K (índices 2–10): ingreso en $ de cada año
- Columnas M–U (índices 13–21): ingreso en $ reales de noviembre 2024
- A partir de la fila 83: mismo bloque pero con errores estándar


In [122]:
wb = load_workbook(excel_path, read_only=True, data_only=True)
ws = wb['3']
filas = list(ws.iter_rows(values_only=True))
wb.close()

regiones = [
    'Arica y Parinacota', 'Tarapacá', 'Antofagasta', 'Atacama',
    'Coquimbo', 'Valparaíso', 'Metropolitana', "O'Higgins",
    'Maule', 'Ñuble', 'Biobío', 'La Araucanía',
    'Los Ríos', 'Los Lagos', 'Aysén', 'Magallanes', 'Total'
]

tipos_ingreso = [
    'Ingreso del trabajo',
    'Ingreso autónomo',
    'Subsidios monetarios',
    'Ingreso monetario',
]

FILA_EST = 9 
FILA_EE  = 86 

registros = []

for i, tipo in enumerate(tipos_ingreso):
    offset = i * 18 

    for k, region in enumerate(regiones):
        fila_est = filas[FILA_EST + offset + k]
        fila_ee  = filas[FILA_EE  + offset + k]

        for j, anio in enumerate(años):
            val_nominal = fila_est[2 + j]
            val_real    = fila_est[14 + j]
            if val_nominal is None and val_real is None:
                continue

            registros.append({
                'anio':anio,
                'tipo_ingreso':tipo,
                'region':region,
                'ingreso_nominal':val_nominal,
                'ingreso_real_nov2024': val_real,
                'error_estandar_nominal':fila_ee[2 + j],
                'error_estandar_real':fila_ee[14 + j],
            })

df_region = pd.DataFrame(registros)

In [123]:
df_region.to_csv(path_carpeta / 'ingreso_region.csv', index=False, encoding='utf-8-sig')

# Ingreso per cápita por decil

Estructura de la hoja:
- Columna A (índice 0): año, aparece solo en la primera fila de cada bloque
- Columna B (índice 1): decil (I al X + Total)
- Columna C (índice 2): media del ingreso per cápita
- Columna D (índice 3): mínimo del ingreso per cápita
- Columna E (índice 4): máximo del ingreso per cápita
- Cada año tiene 11 filas (10 deciles + Total) seguidas de una fila vacía

In [124]:
wb = load_workbook(excel_path, read_only=True, data_only=True)
ws = wb['24']
filas = list(ws.iter_rows(values_only=True))
wb.close()
deciles = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X', 'Total']

registros = []
anio_actual = None

for fila in filas:
    if isinstance(fila[0], int) and 2000 <= fila[0] <= 2030:
        anio_actual = fila[0]
    decil = str(fila[1]).strip() if fila[1] is not None else None

    if decil not in deciles or anio_actual is None:
        continue
    if fila[2] is None:
        continue

    registros.append({
        'anio':anio_actual,
        'decil':decil,
        'ingreso_percapita_media':fila[2],
        'ingreso_percapita_minimo':fila[3],
        'ingreso_percapita_maximo':fila[4],
    })

df_percapita_decil = pd.DataFrame(registros)

In [125]:
df_percapita_decil.to_csv(path_carpeta / 'ingreso_percapita_decil.csv', index=False, encoding='utf-8-sig')

# Filtro Región Metropolitana — CASEN 2024

Este notebook lee los CSV ya limpios y genera versiones filtradas solo con datos de la Región Metropolitana.
Los archivos originales (con todas las regiones) se mantienen intactos.

| CSV original | CSV filtrado RM |
|---|---|
| `ingreso_nacional.csv` | — (es nacional, no tiene región) |
| `ingreso_region.csv` | `ingreso_region_RM.csv` |
| `ingreso_percapita_decil.csv` | — (es nacional, no tiene región) |

In [129]:
CASEN = Path('../datasets/casen_limpia')

archivos_con_region = [
    'ingreso_region.csv',
    'pobreza_region.csv',
]

for nombre in archivos_con_region:
    df = pd.read_csv(CASEN / nombre)

    df_rm = df[df['region'] == 'Metropolitana'].copy()

    nombre_rm = nombre.replace('.csv', '_RM.csv')
    df_rm.to_csv(CASEN / nombre_rm, index=False, encoding='utf-8-sig')